# Power BI Preparation

## Project: Albania Brain Drain Analysis

This notebook prepares dashboard-ready CSV files for Power BI.

The dashboard will include:

1. Executive Summary
2. Migration Analysis
3. Economy and Employment
4. Demographic Risk
5. Forecasting

The purpose of this step is to create clean, simple, dashboard-friendly files that can be imported directly into Power BI.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT_DIR = Path.cwd().parent
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
DASHBOARD_DIR = ROOT_DIR / "dashboard"
POWERBI_DATA_DIR = DASHBOARD_DIR / "powerbi_data"

POWERBI_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Root:", ROOT_DIR)
print("Processed:", PROCESSED_DIR)
print("Dashboard:", DASHBOARD_DIR)
print("Power BI data:", POWERBI_DATA_DIR)

Root: C:\Users\mateo\Downloads\albania-brain-drain-analysis-starter(1)\albania-brain-drain-analysis
Processed: C:\Users\mateo\Downloads\albania-brain-drain-analysis-starter(1)\albania-brain-drain-analysis\data\processed
Dashboard: C:\Users\mateo\Downloads\albania-brain-drain-analysis-starter(1)\albania-brain-drain-analysis\dashboard
Power BI data: C:\Users\mateo\Downloads\albania-brain-drain-analysis-starter(1)\albania-brain-drain-analysis\dashboard\powerbi_data


## Load Main Processed Files

In [2]:
master_demo_path = PROCESSED_DIR / "master_analysis_dataset_with_demographics.csv"
master_path = PROCESSED_DIR / "master_analysis_dataset.csv"

if master_demo_path.exists():
    main = pd.read_csv(master_demo_path)
    print("Using master dataset with demographics")
else:
    main = pd.read_csv(master_path)
    print("Using basic master dataset")

migration_flows = pd.read_csv(PROCESSED_DIR / "migration_flows_country_clean.csv")
residence_reason = pd.read_csv(PROCESSED_DIR / "residence_permits_reason_breakdown_clean.csv")

forecast_path = PROCESSED_DIR / "forecast_results_2035.csv"
forecast_summary_path = PROCESSED_DIR / "forecast_2035_summary.csv"

forecast_results = pd.read_csv(forecast_path) if forecast_path.exists() else pd.DataFrame()
forecast_summary = pd.read_csv(forecast_summary_path) if forecast_summary_path.exists() else pd.DataFrame()

print("Main:", main.shape)
print("Migration flows:", migration_flows.shape)
print("Residence reason:", residence_reason.shape)
print("Forecast results:", forecast_results.shape)
print("Forecast summary:", forecast_summary.shape)

Using master dataset with demographics
Main: (26, 39)
Migration flows: (3859, 8)
Residence reason: (1620, 12)
Forecast results: (93, 5)
Forecast summary: (3, 5)


## Create Main Yearly Dashboard File

In [3]:
main_dashboard_columns = [
    "year",
    "population_total",
    "population_growth_annual_percent",
    "net_migration",
    "net_migration_rate_per_1000",
    "gdp_current_usd",
    "gdp_growth_annual_percent",
    "gdp_per_capita_current_usd",
    "unemployment_total_percent",
    "remittances_percent_gdp",
    "fdi_percent_gdp",
    "total_first_residence_permits",
    "residence_permits_per_100k_population",
    "employment_permit_share",
    "education_permit_share",
    "family_permit_share",
    "migration_pressure_index",
    "youth_population_0_14",
    "working_age_population_15_64",
    "elderly_population_65_plus",
    "youth_share_0_14_percent",
    "working_age_share_15_64_percent",
    "elderly_share_65_plus_percent",
    "dependency_ratio_total",
    "old_age_dependency_ratio",
    "youth_dependency_ratio"
]

available_main_columns = [
    col for col in main_dashboard_columns
    if col in main.columns
]

main_dashboard = main[available_main_columns].copy()

main_dashboard = main_dashboard.sort_values("year")

main_dashboard.to_csv(
    POWERBI_DATA_DIR / "main_yearly_dashboard.csv",
    index=False
)

print("Saved main_yearly_dashboard.csv")
print(main_dashboard.shape)

main_dashboard.head()

Saved main_yearly_dashboard.csv
(26, 26)


,year,population_total,population_growth_annual_percent,net_migration,net_migration_rate_per_1000,gdp_current_usd,gdp_growth_annual_percent,gdp_per_capita_current_usd,unemployment_total_percent,remittances_percent_gdp,...,migration_pressure_index,youth_population_0_14,working_age_population_15_64,elderly_population_65_plus,youth_share_0_14_percent,working_age_share_15_64_percent,elderly_share_65_plus_percent,dependency_ratio_total,old_age_dependency_ratio,youth_dependency_ratio
0,2000,3089027.0,-0.637357,-60531.0,-19.595491,3.584570e+09,7.462859,1160.420471,19.023,16.677034,...,19.595491,914868.0,1947712.0,226447.0,29.616703,63.052606,7.330691,58.597729,11.626308,46.971421
1,2001,3060173.0,-0.938470,-48070.0,-15.708262,4.059064e+09,8.863731,1326.416524,18.570,17.228110,...,15.708262,888582.0,1938841.0,232750.0,29.036989,63.357220,7.605791,57.835171,12.004594,45.830576
2,2002,3051010.0,-0.299877,-45178.0,-14.807556,4.515003e+09,4.628396,1479.838846,17.891,16.247386,...,14.807556,862957.0,1946583.0,241470.0,28.284316,63.801254,7.914429,56.736702,12.404814,44.331888
3,2003,3039616.0,-0.374149,-48517.0,-15.961556,5.801712e+09,5.333264,1908.699007,16.985,15.318730,...,15.961556,835817.0,1953305.0,250494.0,27.497448,64.261587,8.240965,55.613998,12.824111,42.789887
4,2004,3026939.0,-0.417931,-48654.0,-16.073664,7.406646e+09,5.266262,2446.909499,16.306,15.670685,...,16.073664,808130.0,1958916.0,259893.0,26.697933,64.716058,8.586009,54.521123,13.267185,41.253938


## Create Migration Yearly Dashboard File

In [4]:
migration_yearly = (
    migration_flows
    .groupby(["year", "flow_type"], as_index=False)["value"]
    .sum()
)

migration_yearly_wide = (
    migration_yearly
    .pivot_table(
        index="year",
        columns="flow_type",
        values="value",
        aggfunc="sum"
    )
    .reset_index()
)

migration_yearly_wide.columns.name = None

migration_yearly_wide = migration_yearly_wide.rename(columns={
    "immigration": "albanian_citizens_immigrating_to_europe",
    "emigration": "albanian_citizens_emigrating_from_europe"
})

migration_yearly_wide.to_csv(
    POWERBI_DATA_DIR / "migration_yearly_dashboard.csv",
    index=False
)

print("Saved migration_yearly_dashboard.csv")
migration_yearly_wide.head()

Saved migration_yearly_dashboard.csv


,year,albanian_citizens_emigrating_from_europe,albanian_citizens_immigrating_to_europe
0,2015,6078939.0,10169201.0
1,2016,6625508.0,9341215.0
2,2017,6565859.0,9501195.0
3,2018,6169837.0,9780093.0
4,2019,6517560.0,10483989.0


## Create Top Migration Destination File

In [5]:
migration_destinations = (
    migration_flows[
        migration_flows["flow_type"] == "immigration"
    ]
    .groupby(["reporting_country_code", "reporting_country"], as_index=False)["value"]
    .sum()
    .rename(columns={
        "value": "total_immigration_recorded"
    })
    .sort_values("total_immigration_recorded", ascending=False)
)

migration_destinations["rank"] = (
    migration_destinations["total_immigration_recorded"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

migration_destinations.to_csv(
    POWERBI_DATA_DIR / "migration_destinations_dashboard.csv",
    index=False
)

print("Saved migration_destinations_dashboard.csv")
migration_destinations.head(10)

Saved migration_destinations_dashboard.csv


,reporting_country_code,reporting_country,total_immigration_recorded,rank
6,DE,Germany,16828903.0,1
10,ES,Spain,11146851.0,2
12,FR,France,6548079.0,3
34,UK,United Kingdom,6299026.0,4
17,IT,Italy,5578003.0,5
28,PL,Poland,3833529.0,6
30,RO,Romania,3703357.0,7
26,NL,Netherlands,3381532.0,8
3,CH,Switzerland,2514905.0,9
1,BE,Belgium,2261534.0,10


## Create Residence Permits by Reason File

In [6]:
residence_reason_dashboard = (
    residence_reason
    .groupby(["year", "reason_code", "reason"], as_index=False)["value"]
    .sum()
    .rename(columns={
        "value": "total_permits"
    })
    .sort_values(["year", "reason"])
)

yearly_total_subcategories = (
    residence_reason_dashboard
    .groupby("year", as_index=False)["total_permits"]
    .sum()
    .rename(columns={
        "total_permits": "total_reason_subcategory_permits"
    })
)

residence_reason_dashboard = residence_reason_dashboard.merge(
    yearly_total_subcategories,
    on="year",
    how="left"
)

residence_reason_dashboard["reason_share"] = (
    residence_reason_dashboard["total_permits"] /
    residence_reason_dashboard["total_reason_subcategory_permits"]
)

residence_reason_dashboard.to_csv(
    POWERBI_DATA_DIR / "residence_permits_reason_dashboard.csv",
    index=False
)

print("Saved residence_permits_reason_dashboard.csv")
residence_reason_dashboard.head()

Saved residence_permits_reason_dashboard.csv


,year,reason_code,reason,total_permits,total_reason_subcategory_permits,reason_share
0,2008,EDUC,Education reasons,407006,1825803,0.222919
1,2008,EMP,Employment reasons,737366,1825803,0.403858
2,2008,FAM,Family reasons,681431,1825803,0.373223
3,2009,EDUC,Education reasons,470809,1757862,0.267830
4,2009,EMP,Employment reasons,618918,1757862,0.352086


## Create Demographics Dashboard File

In [7]:
demographic_columns = [
    "year",
    "population_total",
    "youth_population_0_14",
    "working_age_population_15_64",
    "elderly_population_65_plus",
    "youth_share_0_14_percent",
    "working_age_share_15_64_percent",
    "elderly_share_65_plus_percent",
    "dependency_ratio_total",
    "old_age_dependency_ratio",
    "youth_dependency_ratio",
    "youth_to_working_age_ratio",
    "elderly_to_working_age_ratio"
]

available_demo_columns = [
    col for col in demographic_columns
    if col in main.columns
]

demographics_dashboard = main[available_demo_columns].copy()

demographics_dashboard.to_csv(
    POWERBI_DATA_DIR / "demographics_dashboard.csv",
    index=False
)

print("Saved demographics_dashboard.csv")
demographics_dashboard.head()

Saved demographics_dashboard.csv


,year,population_total,youth_population_0_14,working_age_population_15_64,elderly_population_65_plus,youth_share_0_14_percent,working_age_share_15_64_percent,elderly_share_65_plus_percent,dependency_ratio_total,old_age_dependency_ratio,youth_dependency_ratio,youth_to_working_age_ratio,elderly_to_working_age_ratio
0,2000,3089027.0,914868.0,1947712.0,226447.0,29.616703,63.052606,7.330691,58.597729,11.626308,46.971421,0.469714,0.116263
1,2001,3060173.0,888582.0,1938841.0,232750.0,29.036989,63.357220,7.605791,57.835171,12.004594,45.830576,0.458306,0.120046
2,2002,3051010.0,862957.0,1946583.0,241470.0,28.284316,63.801254,7.914429,56.736702,12.404814,44.331888,0.443319,0.124048
3,2003,3039616.0,835817.0,1953305.0,250494.0,27.497448,64.261587,8.240965,55.613998,12.824111,42.789887,0.427899,0.128241
4,2004,3026939.0,808130.0,1958916.0,259893.0,26.697933,64.716058,8.586009,54.521123,13.267185,41.253938,0.412539,0.132672


## Create Forecast Dashboard File

In [8]:
if not forecast_results.empty:
    forecast_dashboard = forecast_results.copy()
    
    forecast_dashboard.to_csv(
        POWERBI_DATA_DIR / "forecast_dashboard.csv",
        index=False
    )
    
    print("Saved forecast_dashboard.csv")
    display(forecast_dashboard.head())
else:
    print("Forecast results file not found. Skip for now.")

Saved forecast_dashboard.csv


,indicator,indicator_label,model,year,forecast_value
0,population_total,Population,Moving Average Baseline,2025,2.414286e+06
1,population_total,Population,Moving Average Baseline,2026,2.414286e+06
2,population_total,Population,Moving Average Baseline,2027,2.414286e+06
3,population_total,Population,Moving Average Baseline,2028,2.414286e+06
4,population_total,Population,Moving Average Baseline,2029,2.414286e+06


In [9]:
if not forecast_summary.empty:
    forecast_summary.to_csv(
        POWERBI_DATA_DIR / "forecast_2035_summary_dashboard.csv",
        index=False
    )
    
    print("Saved forecast_2035_summary_dashboard.csv")
    display(forecast_summary.head())
else:
    print("Forecast 2035 summary file not found. Skip for now.")

Saved forecast_2035_summary_dashboard.csv


,indicator,indicator_label,model,year,forecast_value
0,population_total,Population,"ARIMA(1,1,1)",2035,2.197519e+06
1,net_migration,Net Migration,"ARIMA(1,1,1)",2035,-2.427148e+04
2,total_first_residence_permits,First Residence Permits,Moving Average Baseline,2035,2.219822e+06


## Check Created Power BI Files

In [10]:
powerbi_files = sorted(POWERBI_DATA_DIR.glob("*.csv"))

for file in powerbi_files:
    print(file.name)

demographics_dashboard.csv
forecast_2035_summary_dashboard.csv
forecast_dashboard.csv
main_yearly_dashboard.csv
migration_destinations_dashboard.csv
migration_yearly_dashboard.csv
residence_permits_reason_dashboard.csv


In [12]:
from pathlib import Path

ROOT_DIR = Path.cwd().parent
DASHBOARD_DIR = ROOT_DIR / "dashboard"

dax_text = '''
# DAX Measures for Power BI Dashboard

## Main table name

Use this table in Power BI:

main_yearly_dashboard

---

## Population Measures

Total Population =
SUM(main_yearly_dashboard[population_total])

Latest Year =
MAX(main_yearly_dashboard[year])

Latest Population =
CALCULATE(
    SUM(main_yearly_dashboard[population_total]),
    main_yearly_dashboard[year] = MAX(main_yearly_dashboard[year])
)

Population in 2000 =
CALCULATE(
    SUM(main_yearly_dashboard[population_total]),
    main_yearly_dashboard[year] = 2000
)

Population Change =
[Latest Population] - [Population in 2000]

Population Change % =
DIVIDE([Population Change], [Population in 2000])

---

## Migration Measures

Net Migration =
SUM(main_yearly_dashboard[net_migration])

Latest Net Migration =
CALCULATE(
    SUM(main_yearly_dashboard[net_migration]),
    main_yearly_dashboard[year] = MAX(main_yearly_dashboard[year])
)

Migration Rate per 1,000 =
AVERAGE(main_yearly_dashboard[net_migration_rate_per_1000])

Migration Pressure Index =
AVERAGE(main_yearly_dashboard[migration_pressure_index])

---

## Residence Permit Measures

Total Residence Permits =
SUM(main_yearly_dashboard[total_first_residence_permits])

Residence Permits per 100k Population =
AVERAGE(main_yearly_dashboard[residence_permits_per_100k_population])

Employment Permit Share =
AVERAGE(main_yearly_dashboard[employment_permit_share])

Education Permit Share =
AVERAGE(main_yearly_dashboard[education_permit_share])

Family Permit Share =
AVERAGE(main_yearly_dashboard[family_permit_share])

---

## Economy and Employment Measures

GDP Growth =
AVERAGE(main_yearly_dashboard[gdp_growth_annual_percent])

GDP per Capita =
AVERAGE(main_yearly_dashboard[gdp_per_capita_current_usd])

Unemployment Rate =
AVERAGE(main_yearly_dashboard[unemployment_total_percent])

Remittances % GDP =
AVERAGE(main_yearly_dashboard[remittances_percent_gdp])

FDI % GDP =
AVERAGE(main_yearly_dashboard[fdi_percent_gdp])

---

## Demographic Measures

Working Age Population =
SUM(main_yearly_dashboard[working_age_population_15_64])

Elderly Population =
SUM(main_yearly_dashboard[elderly_population_65_plus])

Youth Population =
SUM(main_yearly_dashboard[youth_population_0_14])

Working Age Share =
AVERAGE(main_yearly_dashboard[working_age_share_15_64_percent])

Elderly Share =
AVERAGE(main_yearly_dashboard[elderly_share_65_plus_percent])

Youth Share =
AVERAGE(main_yearly_dashboard[youth_share_0_14_percent])

Dependency Ratio =
AVERAGE(main_yearly_dashboard[dependency_ratio_total])
'''

dax_path = DASHBOARD_DIR / "dax_measures.md"

with open(dax_path, "w", encoding="utf-8") as f:
    f.write(dax_text)

print("Saved:", dax_path)

Saved: C:\Users\mateo\Downloads\albania-brain-drain-analysis-starter(1)\albania-brain-drain-analysis\dashboard\dax_measures.md
